In [10]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

In [11]:
correaltion_data_path

'G:\\XJN_M2project\\correlation_analysis\\test3\\rescue\\5831\\correlation_TST.csv'

In [12]:
# common variables
project_path = projectpath = 'G:\XJN_M2project'
model = 'test3'
condition = 'rescue'
test = 'TST'
condition_path = os.path.join(project_path, 'correlation_analysis', model, condition)
# if model == 'test3':
mice_IDs = [d for d in os.listdir(condition_path) if os.path.isdir(os.path.join(condition_path, d)) and d != 'group']
# else:
#     mice_IDs =  ["CSDS0126", "CSDS0087","CSDS5776","CSDS5797","CSDSQ26","LHQ30","LHQ50","LH0167","LH5798"]
output_path = os.path.join(project_path, 'correlation_analysis', model, condition, 'group', 'depressed')
if os.path.exists(output_path) == False:
    os.makedirs(output_path)

# intial lists to store values from all mice
correlation_values = pd.DataFrame()
Densities = pd.DataFrame()
whole_avg_df = pd.DataFrame()
avg_df = pd.DataFrame()
lccs = pd.DataFrame()
cnps = pd.DataFrame()
degree_df = pd.DataFrame()
CC_df = pd.DataFrame()
BC_df = pd.DataFrame()
# hub_CC_df = pd.DataFrame()
# hub_BC_df = pd.DataFrame()

#main analysis
print('Current time is{}'.format(pd.Timestamp.now()))
for ID in mice_IDs:
    output_path = os.path.join(project_path, 'correlation_analysis', model, condition)
    network_metric_path =os.path.join(project_path, 'correlation_analysis', model, condition, ID,'network_metrics_result_significance.csv')
    correaltion_data_path = os.path.join(project_path, 'correlation_analysis', model, condition, ID, f'correlation_{test}.csv')
    if os.path.exists(network_metric_path) == False:
        print(f'No network metrics file for {ID}, skipping...')
        continue
    if os.path.exists(correaltion_data_path) == False:
        print(f'No correlation data file for {ID}, skipping...')
    network_df = pd.read_csv(network_metric_path)


    ################
    #Average_correlation(all)
    ################
    correaltion_df = pd.read_csv(correaltion_data_path)
    if 'unit_id' in correaltion_df.columns:
        correaltion_df = correaltion_df.drop(columns=['unit_id'])
    if 'Unnamed: 0' in correaltion_df.columns:
        correaltion_df = correaltion_df.drop(columns=['Unnamed: 0'])
    avg_correlation = np.average(correaltion_df)
    whole_avg_df = pd.concat([whole_avg_df, pd.DataFrame({ID: [avg_correlation]})], axis=1)

    ###############
    #Aveerage_correlation(network)
    ###############
    # avg_df = pd.concat([avg_df, pd.DataFrame({ID: [network_df['avg_correlation'].iloc[0]]})], axis=1)

    
    ###############
    #correlation values(all)
    ###############
    upper_triangle_indices = np.triu_indices_from(correaltion_df, k=0)
    linear_data = correaltion_df.values[upper_triangle_indices]
    linear_data = pd.DataFrame({f'Correlation Values{ID}': linear_data})
    correlation_values = pd.concat([correlation_values,linear_data], axis=1)



    ################
    #Network Density
    #N_edges = network_df['Degree']/2
    #N_nodes = network_df['Node'].nunique()
    #Density = N_edges / (N_nodes * (N_nodes - 1)*0.5)
    ###############
    Density = (network_df['Degree'].sum()/2)/ (network_df['Cell_ID'].nunique() * (network_df['Cell_ID'].nunique() - 1) / 2)
    Densities = pd.concat([Densities, pd.DataFrame({ID: [Density]})], axis=1)



    ###############
    #efficient C.C/B.C values
    ##############
    filtered_df = network_df.loc[network_df['Degree'] != 0]
    column_names = ['Degree','Clustering_Coefficient','Betweenness_Centrality']
    filtered_df = filtered_df[column_names]
    filtered_df = filtered_df.reset_index(drop=True)

    normalied_degree = filtered_df[['Degree']] / correaltion_df.shape[0]
    normalied_degree.rename(columns={'Degree': f'Normalized_Degree.{ID}'}, inplace=True)
    degree_df = pd.concat([degree_df, filtered_df[['Degree']],normalied_degree], axis=1)
    degree_df.rename(columns={'Degree': f'Degree.{ID}'}, inplace=True)

    CC_df = pd.concat([CC_df, filtered_df[['Clustering_Coefficient']]], axis=1)
    BC_df = pd.concat([BC_df, filtered_df[['Betweenness_Centrality']]], axis=1)
    CC_df.rename(columns={'Clustering_Coefficient': f'C.C.{ID}'}, inplace=True)
    BC_df.rename(columns={'Betweenness_Centrality': f'B.C.{ID}'}, inplace=True)

    ###############
    # connected component  analysis
    ##############

    # largest connected component
    lcc = network_df['Component_length'].max()
    lccs = pd.concat([lccs, pd.DataFrame({ID: [lcc]})], axis=1)

    #connected node percentage
    cn = np.sum(network_df['Degree']!=0)
    cnp = cn / network_df['Cell_ID'].nunique()
    cnps = pd.concat([cnps, pd.DataFrame({ID: [cnp]})], axis=1)

    ###############
    #hub = node with top 10 degrees
    #hub C.C/B.C values
    ###############

    # hub_df = filtered_df.nlargest(10, 'Degree')
    # hub_df = hub_df.reset_index(drop=True)

    # hub_df = hub_df.sort_values(by='Clustering_Coefficient',ascending=False)
    # hub_CC_df = pd.concat([hub_CC_df, hub_df[['Clustering_Coefficient']]], axis=1)
    # hub_df = hub_df.sort_values(by='Betweenness_Centrality',ascending=False)
    # hub_BC_df = pd.concat([hub_BC_df, hub_df[['Betweenness_Centrality']]], axis=1)

    # hub_CC_df.rename(columns={'Clustering_Coefficient': ID}, inplace=True)    
    # hub_BC_df.rename(columns={'Betweenness_Centrality': ID}, inplace=True)
    combined_df = pd.DataFrame()
    # combined_df = pd.concat([whole_avg_df.T,avg_df.T,Densities.T, lccs.T, cnps.T], axis=1)
    combined_df = pd.concat([whole_avg_df.T,Densities.T, lccs.T, cnps.T], axis=1)
    # combined_df.columns = ['Average_Matrix_correlation','Average_Network_Correlation','Network_Density', 'LCC_size', 'Connected_Node_Percentage']
    combined_df.columns = ['Average_Matrix_correlation','Network_Density', 'LCC_size', 'Connected_Node_Percentage']
    combined_df = combined_df.reset_index()
    combined_df = pd.concat([combined_df,degree_df, CC_df, BC_df], axis=1)

    combined_df.to_csv(os.path.join(output_path,'group', f'{condition}_{test}Network_Metrics_Summary.csv'), index=True, na_rep='')
    # Save individual metric files if needed
    # hub_BC_df.to_csv(os.path.join(output_path, f'{group}_{condition}_{test}_Hub_Betweenness_Centrality.csv'), index=False, na_rep='')
    # hub_CC_df.to_csv(os.path.join(output_path, f'{group}_{condition}_{test}_Hub_Clustering_Coefficient.csv'), index=False, na_rep='')

    print('Network_Metrics_Summary:')
    print( combined_df)
    print(f'Save Results to : {output_path}')


Current time is2026-09-01 15:27:33.188558
Network_Metrics_Summary:
   index  Average_Matrix_correlation  Network_Density  LCC_size  \
0   5806                    0.096116         0.047347      22.0   
1    NaN                         NaN              NaN       NaN   
2    NaN                         NaN              NaN       NaN   
3    NaN                         NaN              NaN       NaN   
4    NaN                         NaN              NaN       NaN   
5    NaN                         NaN              NaN       NaN   
6    NaN                         NaN              NaN       NaN   
7    NaN                         NaN              NaN       NaN   
8    NaN                         NaN              NaN       NaN   
9    NaN                         NaN              NaN       NaN   
10   NaN                         NaN              NaN       NaN   
11   NaN                         NaN              NaN       NaN   
12   NaN                         NaN              NaN       Na